In [1]:
from datasets import load_dataset, Dataset
import json
import os
import random

def _save_jsonl(records, split, subset, seed, num_samples, output_dir="data"):
    os.makedirs(output_dir, exist_ok=True)
    sample_tag = num_samples if num_samples is not None else "all"
    output_path = os.path.join(
        output_dir,
        f"race_{split}_{subset}_{seed}_{sample_tag}.jsonl"
    )

    with open(output_path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    return output_path

def preprocess_race_data(num_samples=None, split="train", subset="high", seed=42, save_jsonl=True):
    dataset_name = "ehovy/race"
    raw_dataset = load_dataset(dataset_name, subset, split=split)

    # Group rows by example_id into one record per article with a questions list.
    grouped = {}
    for row in raw_dataset:
        example_id = row["example_id"]
        if example_id not in grouped:
            grouped[example_id] = {
                "example_id": example_id,
                "article": row["article"],
                "questions": []
            }

        grouped[example_id]["questions"].append({
            "question": row["question"],
            "options": row["options"],
            "answer": row["answer"]
        })

    grouped_examples = list(grouped.values())

    rng = random.Random(seed)
    rng.shuffle(grouped_examples)

    if num_samples is not None:
        grouped_examples = grouped_examples[:min(num_samples, len(grouped_examples))]

    output_path = None
    if save_jsonl:
        output_path = _save_jsonl(
            grouped_examples,
            split=split,
            subset=subset,
            seed=seed,
            num_samples=num_samples
        )

    return Dataset.from_list(grouped_examples), output_path

/storage/project/r-nisha3-0/agupta886/iclr-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset, output_path = preprocess_race_data(num_samples=1000, split="test", subset="high", seed=42)
print("Number of grouped examples:", len(dataset))
print("Saved JSONL:", output_path)
print("\nOne grouped example:\n")
print(dataset[0])

with open(output_path, "r", encoding="utf-8") as f:
    print("\nFirst JSONL line:\n")
    print(f.readline().strip())

Number of grouped examples: 1000
Saved JSONL: data/race_test_high_42_1000.jsonl

One grouped example:

{'example_id': 'high770.txt', 'article': 'Expressions about water are almost as common as water itself.\nThe expressionto be in hot water is a very old expression. Hot water was used five hundred years ago to mean being in trouble. One story says it got that meaning from the custom of throwing extremely hot water down on enemies attacking a castle.\nThat no longer happens. But we still get in hot water. When we are in hot water, we are in trouble. It can be any kind of trouble--serious or not so serious. A person who breaks a law can be in hot water with the police. A young boy can be in hot water with his mother, if he walks in the house with dirty shoes.\nBeingin deep wateris almost the same as being in hot water. When you are in deep water, you are in a difficult position.\nTo keep your head above wateris a colorful expression that means staying out of debt. A company seeks to keep

In [13]:
def create_or_load_preprocessed_data(num_samples=None, split="train", subset="high", seed=42, output_dir="data"):
    sample_tag = num_samples if num_samples is not None else "all"
    output_path = os.path.join(
        output_dir,
        f"race_{split}_{subset}_{seed}_{sample_tag}.jsonl"
    )
    if os.path.exists(output_path):
        print(f"Loading preprocessed data from {output_path}...")
        with open(output_path, "r", encoding="utf-8") as f:
            records = [json.loads(line) for line in f]
        return Dataset.from_list(records), output_path
    else:
        print(f"No preprocessed data found at {output_path}. Preprocessing now...")
        return preprocess_race_data(num_samples=num_samples, split=split, subset=subset, seed=seed, save_jsonl=True)

In [ ]:
dataset, output_path = create_or_load_preprocessed_data(num_samples=500, split="test", subset="high", seed=42)
print("Number of grouped examples:", len(dataset))
print("Saved JSONL:", output_path)
print("\nOne grouped example:\n")
print(dataset[0])

Loading preprocessed data from data/race_train_high_42_3.jsonl...
Number of grouped examples: 3
Saved JSONL: data/race_train_high_42_3.jsonl

One grouped example:

{'example_id': 'high11691.txt', 'article': "Have fun with Stamp collecting...Join the Collectors Club today!If you enjoy learning all about stamps, then the Royal Mail's Collectors Club is for you.Join the club and discover the fascinating world of stamps.There are over 70,000 members and it is one of the biggest clubs of its kind in the country.\nBecome a member today and you will receive lots of wonderful stamp collecting goodies...\n*A Starter pack...\nAnd every two months...\n*Collectors Club magazine\n*Collectors Corner supplement\nAnd every year...\n*A new Stamp Calendar\n*A set of Album Pages\nJoin today!\nIt won't cost you much to experience the fun of stamp collecting.\nCollectors Club Starter Pack\n1.100 used stamps and hinges\n2.The Collectors Club Guide to Collecting\n3.The latest Collectors Club magazine (packed